## Preprocessing

### Table of Contents
- [Dependencies](#dependencies)
- [Load Data & Set Variables](#load-data--set-variables)
- [Define/Categorize Columns](#definecategorize-columns)
- [Fix Inconsistencies](#fix-inconsistencies)
    - [Trim and Lower Case](#trim-and-lower-case-data)
    - [Cast Date Columns to Datetime](#cast-date-columns-to-datetime)
- [Drop Duplicates](#drop-duplicates)
    - [Drop duplicates for data in all columns duplicated](#1-drop-duplicates-for-data-in-all-columns-duplicated)
    - [Drop duplicates per selected columns](#2-drop-duplicates-per-selected-columns)
- [Create Target Column](#create-target-column)
- [Remove and Adjust Records](#remove-and-adjust-records)
    - [Replace Business Type](#replace-business_type)
    - [policy_expiration_date < policy_effective_date](#policy_expiration_date--policy_effective_date)
    - [total_vehicle_count < 0 and prior_year_mileage_000 > 0](#total_vehicle_count--0-and-prior_year_mileage_000--0)
    - [Annual Premium < 0](#annual_premium--0)
- [Add Columns](#add-columns)
  - [Date Columns](#date-columns)
  - ['Vehicle' Columns](#vehicle-columns)
  - [Claim Loss](#claim-loss)
- [Save to parquet](#save-to-parquet)

### Dependencies

In [1]:
import pandas as pd
import numpy as np

from utils import (
    check_duplicates_stats, 
    check_duplicates_per_columns,
)

### Load Data & Set Variables

In [2]:
REAL_DATA = True           # Defines which csv file should be checked/cleaned

train = "./data/orig/train.csv"  # ---- train data ----
real = "./data/orig/score.csv"   # ---- real data ----

In [3]:
if REAL_DATA:
    df = pd.read_csv(real)   # ---- load real data ----
    dataset_type = "real"
else:
    df = pd.read_csv(train) # ---- load train data ----
    dataset_type = "train"

print(f"Dataset type - {dataset_type}:", df.shape)

Dataset type - real: (3000, 28)


### Define/Categorize Columns

In [4]:
ids = ["policy_id", "insured_id"]

not_in_real = ['has_safety_program']

target_columns = [
    "claim_count",
    "target",
    "first_claim_reported_date",
    "claim_paid_amount_current_period",
    "claim_status_current_period",
    "days_to_first_claim_report",
    "total_loss_amount"
]

date_cols = [
    "snapshot_date",
    "policy_effective_date",
    "policy_expiration_date",
    "first_claim_reported_date"
]

target_column = ["target"]

categorical_cols = ['coverage_type', 'business_type', 'state', 'payment_frequency']
cat_cols_direct = ['coverage_type', 'business_type', 'state']
cat_cols_with_unknown = ['payment_frequency']

numeric_cols = [
    'vehicle_count', 'num_heavy_vehicles', 'total_vehicle_count',
    'vehicle_avg_age', 'mileage_per_vehicle',
    'driver_count', 'driver_avg_age',
    'years_in_business',
    'prior_apd_claim_count', 'prior_al_claim_count', 'prior_loss_amount',
    'has_large_prior_loss', 'loss_per_claim',
    'late_payment_count',

    'deductible',
    'coverage_limit_000',
    'annual_premium',
    'risk_score_external', 'risk_score_external_isna',

    'policy_length_days', 'exposure_ratio',
]

## Clean up and Data Preparation

### Fix Inconsistencies
- inconsistent categorical data
    - trim data
    - lower case
- cast date columns to datetime

#### Trim and Lower Case Data

In [5]:
# ---- Trim and Lower Case Data ----

for col in categorical_cols:
    df[col] = df[col].str.strip().str.lower()

print(f'Unique values: {df["business_type"].unique()}')
print(f'Cleaned data: {df.shape}')

Unique values: <ArrowStringArray>
['construction', 'trucking', 'service', 'retail', 'other', 'rideshare']
Length: 6, dtype: str
Cleaned data: (3000, 28)


#### Cast Date Columns to Datetime

In [6]:
# ---- Cast Date Columns to Datetime ----

for col in date_cols:
    df[col] = pd.to_datetime(df[col])

print({df.info()})
print(f'Cleaned data: {df.shape}')

<class 'pandas.DataFrame'>
RangeIndex: 3000 entries, 0 to 2999
Data columns (total 28 columns):
 #   Column                            Non-Null Count  Dtype         
---  ------                            --------------  -----         
 0   policy_id                         3000 non-null   str           
 1   insured_id                        3000 non-null   str           
 2   snapshot_date                     3000 non-null   datetime64[us]
 3   policy_effective_date             3000 non-null   datetime64[us]
 4   policy_expiration_date            3000 non-null   datetime64[us]
 5   coverage_type                     3000 non-null   str           
 6   vehicle_count                     3000 non-null   int64         
 7   vehicle_avg_age                   3000 non-null   float64       
 8   driver_count                      3000 non-null   int64         
 9   driver_avg_age                    2880 non-null   float64       
 10  years_in_business                 3000 non-null   float64  

### Drop Duplicates

In [7]:
check_cols = ['policy_id']

#### 1. Drop Duplicates for Data in All Columns Duplicated

In [8]:
# ---- Drop duplicates for all columns ----
df = df.drop_duplicates()

print(f'Cleaned data: {df.shape}')

Cleaned data: (3000, 28)


#### 2. Drop Duplicates Per Selected Columns

In [9]:
# ---- Drop duplicates per column ----
df.drop_duplicates(subset=['policy_id'], inplace=True)

print(f'Cleaned data: {df.shape}')

Cleaned data: (3000, 28)


#### Recheck duplicates in dataset

In [10]:
check_duplicates_stats(df, check_cols)

for check_col in check_cols:
    check_duplicates_per_columns(df, check_col)

Data contains duplicates: False
Total duplicate rows: 0 

policy_id: False, index: 0


Total duplicated rows for policy_id: 0


### Create Target Column

In [11]:
df["target"] = (df["claim_count"] > 0).astype(int)

KeyError: 'claim_count'

### Remove and Adjust Records
1. update 1 record with 'business_type' == 'agriculture' to 'other'
2. policy_effective_date > policy_expiration_date (10 records)
3. total_vehicle_count == 0 & prior_year_mileage_000 > 0 | driver_count > 0 (2 records)
4. annual_premium < 0 (5 records - assuming those to be cancellations)

#### Replace 'agriculture' with 'other'
- 1 record updated

In [12]:
# ---- Replace 'agriculture' with 'other' ----
# "agriculture" with "other" as all records are from the same target category

df["business_type"] = df["business_type"].replace({"agriculture": "other"})

print(f'Cleaned data: {df.shape}')

Cleaned data: (15000, 32)


#### policy_expiration_date < policy_effective_date
- Drop 10 records

In [11]:
# ---- policy_effective_date > policy_expiration_date ----

df["policy_length_days"] = (df["policy_expiration_date"] - df["policy_effective_date"]).dt.days

In [12]:
df[df["policy_length_days"] < 0]

,policy_id,insured_id,snapshot_date,policy_effective_date,policy_expiration_date,coverage_type,vehicle_count,vehicle_avg_age,driver_count,driver_avg_age,...,annual_premium,payment_frequency,risk_score_external,num_heavy_vehicles,late_payment_count,first_claim_reported_date,claim_paid_amount_current_period,claim_status_current_period,days_to_first_claim_report,policy_length_days


In [14]:
# ---- Drop ----
df = df[df["policy_length_days"] > 0].copy()

print(f'Cleaned data: {df.shape}')

Cleaned data: (14990, 33)


#### total_vehicle_count == 0 and prior_year_mileage_000 > 0
- Drop 2 records

In [13]:
# ---- Check vehicle_count == 0 & num_heavy_vehicles == 0 & prior_year_mileage_000 > 0 ----

invalid_vehicle_mask = ( 
    (df["vehicle_count"] == 0) & (df["num_heavy_vehicles"] == 0)
    ) & (
    (df["prior_year_mileage_000"] > 0) | (df["driver_count"] > 0)
)
display(df[invalid_vehicle_mask])

,policy_id,insured_id,snapshot_date,policy_effective_date,policy_expiration_date,coverage_type,vehicle_count,vehicle_avg_age,driver_count,driver_avg_age,...,annual_premium,payment_frequency,risk_score_external,num_heavy_vehicles,late_payment_count,first_claim_reported_date,claim_paid_amount_current_period,claim_status_current_period,days_to_first_claim_report,policy_length_days


In [16]:
# ---- Drop 2 rows ----
df = df[~invalid_vehicle_mask].copy()

print(f'Cleaned data: {df.shape}')

Cleaned data: (14988, 33)


#### annual_premium < 0
- Drop 5 records

In [14]:
df[df["annual_premium"] < 0 ]

,policy_id,insured_id,snapshot_date,policy_effective_date,policy_expiration_date,coverage_type,vehicle_count,vehicle_avg_age,driver_count,driver_avg_age,...,annual_premium,payment_frequency,risk_score_external,num_heavy_vehicles,late_payment_count,first_claim_reported_date,claim_paid_amount_current_period,claim_status_current_period,days_to_first_claim_report,policy_length_days


In [18]:
df = df[df["annual_premium"] >= 0 ].copy()
df.shape

(14983, 33)

### Add Columns
Date Columns:
1. elapsed_days = snapshot_date - policy_effective_date
2. exposure_days = elapsed_days.clip(lower=1.0) -> if 0 put to 1
3. exposure_ratio = elapsed_days / policy_length_days                       -> use
4. remaining_days = policy_expiration_date - snapshot_date

Vehicle Columns:

5. total_vehicle_count = vehicle_count + num_heavy_vehicles
6. mileage_per_vehicle = prior_year_mileage_000 / total_vehicle_count

Claim Loss:

7. loss_per_claim = if total_prior_claims > 0 then prior_loss_amount / total_prior_claims else 0.0
8. has_large_prior_loss = prior_loss_amount > 50000
9. risk_score_external_isna = df["risk_score_external"].isna().astype(int)

#### Date Columns

In [15]:
# ---- calculate elapsed days ----
df["elapsed_days"] = (df["snapshot_date"] - df["policy_effective_date"]).dt.days

# ---- calculate exposure days with floor ---- 
df["exposure_days"] = df["elapsed_days"].clip(lower=1.0)

# ---- calculate ratio between elapsed_days and policy_length_days----
df["exposure_ratio"] = df["elapsed_days"] / df["policy_length_days"].clip(
    lower=0.0, upper=1.0)

# ---- calculate remaining days ----
df["remaining_days"] = (df["policy_expiration_date"] - df["snapshot_date"]).dt.days

df[["policy_length_days", "elapsed_days", "exposure_days", "exposure_ratio", "remaining_days"]].head()

,policy_length_days,elapsed_days,exposure_days,exposure_ratio,remaining_days
0,365,79,79,79.0,286
1,365,166,166,166.0,199
2,365,31,31,31.0,334
3,365,142,142,142.0,223
4,365,68,68,68.0,297


#### Vehicle Columns

In [16]:
# ---- total_vehicle_count ----
df['total_vehicle_count'] = df['vehicle_count'] + df['num_heavy_vehicles']
df[["vehicle_count","num_heavy_vehicles", "total_vehicle_count"]].head()

,vehicle_count,num_heavy_vehicles,total_vehicle_count
0,10,4,14
1,9,3,12
2,5,2,7
3,12,1,13
4,9,2,11


In [17]:
# ---- mileage_per_vehicle ----
df['mileage_per_vehicle'] = np.where(
    df['total_vehicle_count'] > 0, 
    df['prior_year_mileage_000'] / df['total_vehicle_count'], 
    np.nan
)

df['mileage_per_vehicle'].head()

0    19.378571
1    33.941667
2    19.685714
3    18.907692
4    38.118182
Name: mileage_per_vehicle, dtype: float64

#### Claim Loss

In [18]:
# ---- loss_per_claim ----
total_prior_claims = df["prior_apd_claim_count"] + df["prior_al_claim_count"]

df['loss_per_claim'] = np.where(
    total_prior_claims > 0,
    df['prior_loss_amount'] / total_prior_claims, 
    0.0
)

df['loss_per_claim'].head()

0        0.00
1    12195.65
2     6682.99
3     8650.00
4     6484.67
Name: loss_per_claim, dtype: float64

In [19]:
# ---- has_large_prior_loss ----
df["has_large_prior_loss"] = (df["prior_loss_amount"] > 50000).astype(int)

df["has_large_prior_loss"].value_counts()

has_large_prior_loss
0    2808
1     192
Name: count, dtype: int64

In [20]:
# ---- risk_score_external_isna ----
df["risk_score_external_isna"] = (
    df["risk_score_external"].isna().astype(int)
)

df["risk_score_external_isna"].value_counts()

risk_score_external_isna
0    2850
1     150
Name: count, dtype: int64

### Save to parquet

In [21]:
df.to_parquet(f'./data/cleaned/df_{dataset_type}_cleaned.parquet')